In [1]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# 🔹 Надёжный путь к файлу (поднимаемся из notebooks/ в корень проекта)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
CSV_PATH = os.path.join(PROJECT_ROOT, 'data', 'raw', 'logistics_data.csv')

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"❌ Файл не найден: {CSV_PATH}\n💡 Сначала запусти '01_data_generation_eda.ipynb'")

# Загрузка
df = pd.read_csv(CSV_PATH)
df['order_date'] = pd.to_datetime(df['order_date'])
print(f"✅ Загружено {len(df):,} строк из: {CSV_PATH}")

# Feature Engineering
df['day_of_week'] = df['order_date'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['heavy_package'] = (df['weight_kg'] > 5).astype(int)
df['long_distance'] = (df['distance_km'] > 300).astype(int)
df['is_delayed'] = (df['delay_days'] > 1).astype(int)

# Подготовка X, y
features = ['distance_km', 'weight_kg', 'day_of_week', 'is_weekend', 'heavy_package', 'long_distance']
categorical = ['carrier', 'warehouse_id', 'destination_city']
X = pd.get_dummies(df[features + categorical], drop_first=True)
y = df['is_delayed']

# Разделение
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Обучение
model = RandomForestClassifier(n_estimators=100, max_depth=5, class_weight='balanced', random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Оценка
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("\n📊 Оценка модели:")
print(classification_report(y_test, y_pred, zero_division=0))
print(f"📈 ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}\n")

importance = pd.DataFrame({'feature': X.columns, 'importance': model.feature_importances_})
print("🔑 Топ-5 факторов задержки:")
print(importance.sort_values('importance', ascending=False).head(5).to_string(index=False))

✅ Загружено 10,000 строк из: c:\Users\user\Documents\GitHub\logistics-analytics\data\raw\logistics_data.csv

📊 Оценка модели:
              precision    recall  f1-score   support

           0       0.92      0.70      0.80      1823
           1       0.10      0.34      0.15       177

    accuracy                           0.67      2000
   macro avg       0.51      0.52      0.47      2000
weighted avg       0.84      0.67      0.74      2000

📈 ROC-AUC: 0.498

🔑 Топ-5 факторов задержки:
          feature  importance
      distance_km    0.298171
        weight_kg    0.242569
      day_of_week    0.098320
carrier_PickPoint    0.051658
     carrier_CDEK    0.040734


In [2]:
import plotly.express as px

route_delay = df.groupby(['warehouse_id', 'destination_city']).agg(
    avg_delay=('delay_days', 'mean')
).reset_index().dropna()

fig = px.density_heatmap(
    route_delay, x='destination_city', y='warehouse_id', z='avg_delay',
    text_auto='.1f', title='🗺️ Средняя задержка (дней) по маршрутам',
    color_continuous_scale='RdYlGn_r'
)
fig.show()